# GraphSAGE Baseline — Elliptic Bitcoin Fraud Detection

This notebook trains a two-layer GraphSAGE node classifier using the **same preprocessed graph, temporal masks, labels, class weights, and validation-F1 checkpoint rule** as the stabilized GCN baseline.

- Label `0`: legitimate; label `1`: illicit/fraud; label `-1`: unknown (excluded from supervised masks).
- This is full-graph, transductive training to keep the comparison consistent with the existing GCN.
- The test set is used only for final evaluation after selecting the checkpoint on validation F1.
- This notebook does not modify the GCN notebook or the Streamlit app.

## 1. Imports and reproducibility

In [ ]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_curve,
    roc_curve,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Resolve project paths and load the existing graph

In [ ]:
# Supports running Jupyter with either the repository root or model/ as the working directory.
cwd = Path.cwd().resolve()
if (cwd / "data" / "processed").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data" / "processed").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate data/processed from the current working directory: "
        f"{cwd}. Run this notebook from the repository root or model/."
    )

DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "model"
GRAPH_PATH = DATA_DIR / "elliptic_graph.pt"
CLASS_WEIGHTS_PATH = DATA_DIR / "class_weights.pt"
CHECKPOINT_PATH = MODEL_DIR / "best_graphsage_model.pt"
METRICS_PATH = MODEL_DIR / "graphsage_metrics.json"
RESULTS_DIR = PROJECT_ROOT / "results" / "GraphSAGE"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for required_path in (GRAPH_PATH, CLASS_WEIGHTS_PATH):
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required file not found: {required_path}\n"
            "Run notebooks/elliptic_preprocessing.ipynb locally first. "
            "Large processed data files are intentionally not required in Git."
        )

graph_data = torch.load(
    GRAPH_PATH,
    map_location="cpu",
    weights_only=False,
)

print(graph_data)
print("Features:", tuple(graph_data.x.shape))
print("Edges:", tuple(graph_data.edge_index.shape))
print("Train nodes:", int(graph_data.train_mask.sum()))
print("Validation nodes:", int(graph_data.val_mask.sum()))
print("Test nodes:", int(graph_data.test_mask.sum()))
print("Unique labels:", torch.unique(graph_data.y).tolist())

## 3. Validate masks and labels

In [ ]:
# Supervised masks must contain only known binary labels.
for split_name in ("train", "val", "test"):
    mask = getattr(graph_data, f"{split_name}_mask")
    labels = graph_data.y[mask]
    assert mask.dtype == torch.bool
    assert labels.numel() > 0, f"{split_name} mask is empty"
    assert bool(torch.all((labels == 0) | (labels == 1))), (
        f"{split_name} mask includes labels outside {{0, 1}}"
    )

train_labels = graph_data.y[graph_data.train_mask]
print("Training class counts:", torch.bincount(train_labels, minlength=2).tolist())
print("All supervised masks exclude unknown label -1.")

## 4. Define the two-layer GraphSAGE model

In [ ]:
class GraphSAGE(torch.nn.Module):
    def __init__(self, num_features, hidden_channels=64, num_classes=2, dropout=0.5):
        super().__init__()
        self.conv1 = SAGEConv(num_features, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x  # raw logits; CrossEntropyLoss applies the appropriate log-softmax

num_features = graph_data.num_node_features
hidden_channels = 64
num_classes = 2

model = GraphSAGE(
    num_features=num_features,
    hidden_channels=hidden_channels,
    num_classes=num_classes,
).to(device)

graph_data = graph_data.to(device)
class_weights = torch.load(
    CLASS_WEIGHTS_PATH,
    map_location=device,
    weights_only=True,
).float()

assert class_weights.numel() == num_classes
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=5e-4,
)

print(model)
print("Class weights:", class_weights.detach().cpu().tolist())

## 5. Training and validation helpers

In [ ]:
@torch.no_grad()
def predict_all():
    model.eval()
    return model(graph_data.x, graph_data.edge_index)

@torch.no_grad()
def evaluate(mask, logits=None):
    if logits is None:
        logits = predict_all()

    y_true = graph_data.y[mask].detach().cpu().numpy()
    split_logits = logits[mask]
    y_pred = split_logits.argmax(dim=1).detach().cpu().numpy()
    fraud_probs = torch.softmax(split_logits, dim=1)[:, 1].detach().cpu().numpy()

    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "fraud_precision": float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "fraud_recall": float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "fraud_f1": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "pr_auc_average_precision": float(average_precision_score(y_true, fraud_probs)),
    }

    # ROC-AUC is undefined if a split contains only one class.
    metrics["roc_auc"] = (
        float(roc_auc_score(y_true, fraud_probs))
        if np.unique(y_true).size == 2
        else None
    )
    return metrics

def train_one_epoch():
    model.train()
    optimizer.zero_grad()
    logits = model(graph_data.x, graph_data.edge_index)
    loss = criterion(
        logits[graph_data.train_mask],
        graph_data.y[graph_data.train_mask],
    )
    loss.backward()
    optimizer.step()
    return float(loss.item())

## 6. Train and select the checkpoint using validation fraud F1

In [ ]:
NUM_EPOCHS = 100
best_val_f1 = -1.0
train_losses = []
val_f1_scores = []
val_pr_auc_scores = []

MODEL_DIR.mkdir(parents=True, exist_ok=True)

for epoch in range(1, NUM_EPOCHS + 1):
    loss = train_one_epoch()
    logits = predict_all()
    val_metrics = evaluate(graph_data.val_mask, logits=logits)

    train_losses.append(loss)
    val_f1_scores.append(val_metrics["fraud_f1"])
    val_pr_auc_scores.append(val_metrics["pr_auc_average_precision"])

    # Keep the same checkpoint-selection criterion as the GCN baseline.
    if val_metrics["fraud_f1"] > best_val_f1:
        best_val_f1 = val_metrics["fraud_f1"]
        torch.save(model.state_dict(), CHECKPOINT_PATH)

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | Loss {loss:.4f} | "
            f"Val Precision {val_metrics['fraud_precision']:.4f} | "
            f"Val Recall {val_metrics['fraud_recall']:.4f} | "
            f"Val F1 {val_metrics['fraud_f1']:.4f} | "
            f"Val PR-AUC {val_metrics['pr_auc_average_precision']:.4f}"
        )

print(f"Best validation fraud F1: {best_val_f1:.4f}")
print("Checkpoint saved to:", CHECKPOINT_PATH)

## 7. Load the best checkpoint and run final test evaluation

In [ ]:
# Restore the validation-selected checkpoint before the one-time test evaluation.
best_state = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
model.load_state_dict(best_state)
model.eval()

test_logits = predict_all()
test_metrics = evaluate(graph_data.test_mask, logits=test_logits)

y_test = graph_data.y[graph_data.test_mask].detach().cpu().numpy()
test_predictions = test_logits[graph_data.test_mask].argmax(dim=1).detach().cpu().numpy()

print("Final test metrics:")
print(json.dumps(test_metrics, indent=2))
print("\nClassification report (0 = Legitimate, 1 = Fraud):")
print(
    classification_report(
        y_test,
        test_predictions,
        labels=[0, 1],
        target_names=["Legitimate", "Fraud"],
        zero_division=0,
        digits=4,
    )
)

## 8. Confusion matrix and training curves

In [ ]:
# Save publication-quality GraphSAGE plots to results/GraphSAGE/.
cm = confusion_matrix(y_test, test_predictions, labels=[0, 1])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Legitimate", "Fraud"],
)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, values_format="d", colorbar=False)
ax.set_title("GraphSAGE — Test Confusion Matrix")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# Training loss
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(train_losses) + 1), train_losses)
ax.set_title("GraphSAGE — Training Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Weighted cross-entropy loss")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "training_loss.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# Validation fraud F1
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(val_f1_scores) + 1), val_f1_scores)
ax.set_title("GraphSAGE — Validation Fraud F1")
ax.set_xlabel("Epoch")
ax.set_ylabel("Fraud F1")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "validation_f1.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# Validation PR-AUC over epochs
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(val_pr_auc_scores) + 1), val_pr_auc_scores)
ax.set_title("GraphSAGE — Validation PR-AUC")
ax.set_xlabel("Epoch")
ax.set_ylabel("Average precision")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "validation_pr_auc.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# Test precision-recall and ROC curves use probability scores, not hard predictions.
test_probabilities = torch.softmax(test_logits[graph_data.test_mask], dim=1)[:, 1].detach().cpu().numpy()
pr_precision, pr_recall, _ = precision_recall_curve(y_test, test_probabilities)
pr_auc_value = average_precision_score(y_test, test_probabilities)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(pr_recall, pr_precision, label=f"Average precision = {pr_auc_value:.4f}")
ax.set_title("GraphSAGE — Test Precision-Recall Curve")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "precision_recall_curve.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

roc_fpr, roc_tpr, _ = roc_curve(y_test, test_probabilities)
roc_auc_value = roc_auc_score(y_test, test_probabilities)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(roc_fpr, roc_tpr, label=f"ROC-AUC = {roc_auc_value:.4f}")
ax.plot([0, 1], [0, 1], linestyle="--", label="Random baseline")
ax.set_title("GraphSAGE — Test ROC Curve")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "roc_curve.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"Saved GraphSAGE figures to: {RESULTS_DIR}")

## 9. Save metrics for the model comparison

In [ ]:
metrics_payload = {
    "model": "GraphSAGE",
    "dataset": "Elliptic Bitcoin",
    "seed": SEED,
    "architecture": {
        "layers": 2,
        "hidden_channels": hidden_channels,
        "dropout": 0.5,
        "aggregation": "mean (PyG SAGEConv default)",
    },
    "training": {
        "epochs": NUM_EPOCHS,
        "learning_rate": 0.01,
        "weight_decay": 5e-4,
        "checkpoint_selection": "best validation fraud F1",
        "best_validation_fraud_f1": float(best_val_f1),
    },
    "test_metrics": test_metrics,
    "split_sizes": {
        "train": int(graph_data.train_mask.sum().item()),
        "validation": int(graph_data.val_mask.sum().item()),
        "test": int(graph_data.test_mask.sum().item()),
    },
}

with METRICS_PATH.open("w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

# Keep a copy with the experiment artifacts for cross-model comparison.
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with (RESULTS_DIR / "metrics.json").open("w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)
pd.DataFrame([test_metrics]).to_csv(RESULTS_DIR / "metrics.csv", index=False)

print("Metrics saved to:", METRICS_PATH)
print("Checkpoint:", CHECKPOINT_PATH)
print("\nReminder: compare this model against GCN using the exact same graph and masks.")

## Notes for the comparison

- Compare fraud-class precision, recall, F1, PR-AUC (average precision), ROC-AUC, and the confusion matrix. Accuracy alone is not sufficient for this imbalanced dataset.
- These scores are comparable to the GCN baseline only if both checkpoints use the same graph artifact, masks, and evaluation definitions.
- GraphSAGE's checkpoint is saved separately as `model/best_graphsage_model.pt`; it does not overwrite `best_gcn_model.pt`.
- The Streamlit loader is still GCN-specific. Do not select this checkpoint in the current dashboard until the model-loading and explainer code are deliberately generalized and tested.

## Research output files

The plotting cell above writes high-resolution PNG figures to `results/GraphSAGE/`. The metrics cell also writes `metrics.json` and `metrics.csv` there, while retaining the existing `model/graphsage_metrics.json` output.